# 02 - Formula 1 Spark Optimization Settings

**Focus:** only the Spark configuration settings that directly support the optimization demos.

This notebook is intentionally configuration-focused. Do not teach dozens of Spark properties.

## Priority settings
- `spark.sql.shuffle.partitions`
- `spark.sql.adaptive.enabled`
- `spark.sql.adaptive.coalescePartitions.enabled`
- `spark.sql.adaptive.advisoryPartitionSizeInBytes`
- `spark.sql.adaptive.localShuffleReader.enabled`
- `spark.sql.adaptive.skewJoin.enabled`
- skew thresholds
- `spark.sql.autoBroadcastJoinThreshold`
- `spark.sql.adaptive.autoBroadcastJoinThreshold`
- `spark.sql.files.maxPartitionBytes`
- `spark.sql.files.openCostInBytes`

**Important:** show settings, run a small Formula 1 query, inspect the plan, then explain the purpose. Do not claim a setting automatically makes every workload faster.

In [0]:
from pyspark.sql import functions as F

## 1. Read current settings

In [0]:
settings = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.localShuffleReader.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
    "spark.sql.files.maxPartitionBytes",
    "spark.sql.files.openCostInBytes"
]

for key in settings:
    try:
        print(f"{key} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key} = <not explicitly available>")

# 2. Shuffle settings

`spark.sql.shuffle.partitions` controls the number of shuffle partitions used by shuffle operations.

Demonstrate a small value, then inspect the physical plan.

In [0]:
results_df = spark.table("formula1_dev.silver.results")
races_df = spark.table("formula1_dev.silver.races")

demo_races_df = races_df.filter(F.col("race_year").isin([2020, 2021]))
demo_results_df = results_df.join(
    demo_races_df.select("race_id").distinct(),
    "race_id",
    "inner"
)

spark.conf.set("spark.sql.shuffle.partitions", 8)

shuffle_df = (
    demo_results_df
    .groupBy("driver_id")
    .agg(F.sum("points").alias("total_points"))
)

shuffle_df.explain(True)

### Teaching point

Too few partitions → larger tasks.
Too many → many small tasks and scheduling overhead.
There is no universal magic number.

# 3. AQE settings

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Coalesce:", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))
print("Local shuffle reader:", spark.conf.get("spark.sql.adaptive.localShuffleReader.enabled"))
print("Skew join:", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

## 4. AQE advisory partition size

In [0]:
spark.conf.set(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    64 * 1024 * 1024
)

print(
    "AQE advisory partition size:",
    spark.conf.get("spark.sql.adaptive.advisoryPartitionSizeInBytes")
)

This is a target size used by AQE when it optimizes shuffle partition sizing.
It is not a command to create exactly 64 MB files.

# 5. Broadcast settings

In [0]:
print(
    "Normal broadcast threshold:",
    spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
)

try:
    print(
        "AQE broadcast threshold:",
        spark.conf.get("spark.sql.adaptive.autoBroadcastJoinThreshold")
    )
except Exception:
    print("AQE broadcast threshold is not explicitly configured.")

## Formula 1 broadcast demo

In [0]:
drivers_df = spark.table("formula1_dev.silver.drivers")

broadcast_df = (
    demo_results_df.alias("r")
    .join(
        F.broadcast(drivers_df).alias("d"),
        F.col("r.driver_id") == F.col("d.driver_id"),
        "inner"
    )
)

broadcast_df.explain(True)

Look for `BroadcastHashJoin`.

The threshold is a **size-based decision aid**. The exact join selected can also be affected by AQE and the available statistics.

# 6. Skew settings

In [0]:
spark.conf.set(
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    64 * 1024 * 1024
)

spark.conf.set(
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    5
)

print(
    "Skew threshold:",
    spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes")
)
print(
    "Skew factor:",
    spark.conf.get("spark.sql.adaptive.skewJoin.skewedPartitionFactor")
)

**Concept:**

A partition can be considered skewed when it is sufficiently large and significantly larger than the median partition, subject to the runtime's skew logic.

# 7. File input settings

In [0]:
print(4194304 / (1024 * 1024), "MB")

In [0]:
print(
    "maxPartitionBytes:",
    spark.conf.get("spark.sql.files.maxPartitionBytes")
)

print(
    "openCostInBytes:",
    spark.conf.get("spark.sql.files.openCostInBytes")
)

`maxPartitionBytes` influences how much input data Spark groups into an input partition.
`openCostInBytes` models the cost of opening a file when Spark combines files into input partitions.

These are useful when discussing many files in ADLS/Delta.

# 8. One clean configuration block for a demo

Use this as the controlled classroom configuration. Values are deliberately modest for demonstration.

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.localShuffleReader.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

spark.conf.set(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    64 * 1024 * 1024
)

# 9. What NOT to tune casually

Do not turn this notebook into a list of:

- executor memory
- executor cores
- driver memory
- memory fractions
- dozens of low-level JVM settings

On managed Databricks compute, these are often controlled at the compute/runtime level.
Teach them separately under cluster sizing if needed.

# Final settings map

```text
Shuffle problem
    → spark.sql.shuffle.partitions

Runtime adaptation
    → spark.sql.adaptive.*

Small dimension join
    → spark.sql.autoBroadcastJoinThreshold

Skewed join
    → spark.sql.adaptive.skewJoin.*

File-read behavior
    → spark.sql.files.*
```